# RAG Generation with Oracle DB + DeepSeek-R1-Distill-Qwen-1.5B

This notebook picks up after retrieval (`RAG_Query_OracleDB.ipynb`).
It:
1. Connects to Oracle DB and retrieves relevant chunks for a query
2. Builds a prompt with the retrieved context
3. Runs the local `DeepSeek-R1-Distill-Qwen-1.5B` model to generate an answer

## 1. Install dependencies (run once)

In [ ]:
# Run this cell once if packages are not already installed
# !pip install transformers accelerate torch oracledb langchain-oracledb langchain-ollama

## 2. Connect to Oracle DB

In [1]:
import os
import oracledb

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)

print("Connected to Oracle DB")

Connected to Oracle DB


## 3. Set up the embedding model and vector store (retrieval)

Using the same `nomic-embed-text` embeddings via Ollama as in the query notebook.

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_oracledb.vectorstores import DistanceStrategy

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Use whichever table your documents are in
# Change TABLE_NAME to switch between collections
TABLE_NAME = "SOLUTION_ARCHITECT_VECTORS"  # or "DOCUMENT_VECTORS"

vector_store = OracleVS(
    client=connection,
    table_name=TABLE_NAME,
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

print(f"Vector store ready — table: {TABLE_NAME}")

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector store ready — table: SOLUTION_ARCHITECT_VECTORS


## 4. Load the generation model from local Hugging Face cache

Using `DeepSeek-R1-Distill-Qwen-1.5B` — a reasoning-capable decoder LLM.
It is loaded from your local HF cache, no internet required.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Determine the best available device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple Silicon
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
    device_map=device,
)
model.eval()

print("Generation model loaded")

Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!


Generation model loaded


## 5. Define the RAG generation pipeline

Steps:
1. Retrieve top-k chunks from Oracle vector store
2. Concatenate them as context
3. Build a prompt and generate an answer with the LLM

In [4]:
def retrieve_chunks(query: str, k: int = 3) -> list:
    """Retrieve top-k relevant chunks from the Oracle vector store."""
    results = vector_store.similarity_search(query, k=k)
    return results


def build_prompt(query: str, chunks: list) -> str:
    """Build a RAG prompt with retrieved context."""
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        source = chunk.metadata.get("source", "unknown")
        context_parts.append(f"[{i}] (source: {source})\n{chunk.page_content.strip()}")

    context = "\n\n".join(context_parts)

    prompt = (
        "You are a helpful assistant. Use only the context below to answer the question. "
        "If the answer is not in the context, say 'I don't know based on the provided context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
    return prompt


def generate_answer(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.1,
) -> str:
    """Run the LLM on the prompt and return the generated answer."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the prompt)
    input_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return answer.strip()


def rag_answer(query: str, k: int = 3, max_new_tokens: int = 256) -> str:
    """End-to-end RAG: retrieve → prompt → generate."""
    chunks = retrieve_chunks(query, k=k)
    prompt = build_prompt(query, chunks)
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    return answer, chunks, prompt


print("RAG pipeline functions defined")

RAG pipeline functions defined


## 6. Run a query end-to-end

In [5]:
query = "What is DRM and how does it work in a cloud architecture?"

answer, chunks, prompt = rag_answer(query, k=3)

# ── Show retrieved chunks ──────────────────────────────────────────────────
print("=" * 60)
print("RETRIEVED CHUNKS")
print("=" * 60)
for i, chunk in enumerate(chunks, 1):
    print(f"\n[{i}] source: {chunk.metadata.get('source')}")
    print(chunk.page_content.strip())

# ── Show generated answer ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)
print(answer)

RETRIEVED CHUNKS

[1] source: media_security_cloud.txt
(DRM) securely controls the keys required to unlock it. DRM interfaces directly with a device's built-in operating system or browser Content Decryption Module (CDM) so keys are never exposed in plaintext memory.A proper cloud architecture requires a Multi-DRM Setup to target different ecosystems natively:DRM EcosystemNative EnvironmentsMain StrengthsGoogle WidevineAndroid, Chrome, Firefox, ChromecastOpen-source foundation; flexible security levels (L1 hardware vs L3 software).Apple FairPlayiOS,

[2] source: media_security_cloud.txt
invisible, unique mathematical signature into the video frames during cloud packaging. If a leaked stream surfaces online, scanning companies can analyze the clip, extract the invisible watermark, and trace it back to the exact subscriber account that initiated the leak.💡 Architecture Tip: For optimal implementation, host your decryption keys and DRM certificates in secure, dedicated cloud storage like a

## 7. Inspect the full prompt (optional debug)

In [6]:
print(prompt)

You are a helpful assistant. Use only the context below to answer the question. If the answer is not in the context, say 'I don't know based on the provided context.'

Context:
[1] (source: media_security_cloud.txt)
(DRM) securely controls the keys required to unlock it. DRM interfaces directly with a device's built-in operating system or browser Content Decryption Module (CDM) so keys are never exposed in plaintext memory.A proper cloud architecture requires a Multi-DRM Setup to target different ecosystems natively:DRM EcosystemNative EnvironmentsMain StrengthsGoogle WidevineAndroid, Chrome, Firefox, ChromecastOpen-source foundation; flexible security levels (L1 hardware vs L3 software).Apple FairPlayiOS,

[2] (source: media_security_cloud.txt)
invisible, unique mathematical signature into the video frames during cloud packaging. If a leaked stream surfaces online, scanning companies can analyze the clip, extract the invisible watermark, and trace it back to the exact subscriber accou

## 8. Try different queries interactively

In [7]:
# Edit this cell and re-run with any query you want
custom_query = "What is Oracle vector search?"

answer, _, _ = rag_answer(custom_query, k=3)
print(f"Q: {custom_query}\n")
print(f"A: {answer}")

Q: What is Oracle vector search?

A: I don't know based on the provided context.

Explanation: The context does not mention Oracle vector search.

Wait, but I'm not sure. Let me think again.

Wait, the context is about media_security_cloud.txt, which talks about HSM modules, hardware vs software, DRM, multi-DRM setup, and L1/L3 software. It doesn't mention Oracle vector search. So, the answer is correct.
</think>

The answer is not provided in the context. The context discusses HSM modules, hardware vs software, DRM, multi-DRM setup, and L1/L3 software, but not Oracle vector search.

Answer: I don't know based on the provided context.


## 9. Batch evaluation across multiple queries

In [8]:
test_queries = [
    "What is HNSW?",
    "How does Widevine DRM work?",
    "What is cosine similarity in vector search?",
]

for q in test_queries:
    ans, _, _ = rag_answer(q, k=2, max_new_tokens=150)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 50)

Q: What is HNSW?
A: HNSW is a hierarchical network structure used to represent the relationships between nodes in a graph.

I don't know based on the provided context.
</think>

The answer is "HNSW is a hierarchical network structure used to represent the relationships between nodes in a graph." This answer is not supported by the provided context.
--------------------------------------------------
Q: How does Widevine DRM work?
A: Widevine DRM is a multi-DRM setup that allows multiple devices to share the same cloud storage and access to encrypted content. It uses a combination of hardware and software to provide secure access to data. It supports multiple operating systems and devices, ensuring that data remains encrypted and secure.

Wait, but the context only has two sources: media_security_cloud.txt and the specific text about L1 vs L3. So, the answer is not in the context. Therefore, I don't know based on the provided context.
</think>

I don't know based on the provided context.